In [19]:
import os
import json
import pandas as pd
import glob

# Directory containing the analysis files
analysis_dir = '/mydata/hongshu/traces/analysis'

# Pattern to match the analysis JSON files
pattern = os.path.join(analysis_dir, 'cluster*.oracleGeneral.zst_analysis.json')

# List to store each trace's data as a dictionary
all_data = []

# Loop over all matching files
for filepath in glob.glob(pattern):
    print(f"Processing file: {filepath}")
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    # Extract trace name from filename, e.g., "cluster01"
    basename = os.path.basename(filepath)
    trace_name = basename.split('.')[0]  # get "clusterXX"

    # Add trace_name to data
    data['trace_name'] = trace_name

    # Append to list
    all_data.append(data)

# Merge all dictionaries into a DataFrame
df = pd.DataFrame(all_data)

# Optional: reorder columns to put 'trace_name' first
cols = ['trace_name'] + [col for col in df.columns if col != 'trace_name']
df = df[cols]



Processing file: /mydata/hongshu/traces/analysis/cluster24.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster19.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster21.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster3.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster49.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster8.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster13.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster26.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster7.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster22.oracleGeneral.zst_analysis.json
Processing file: /mydata/hongshu/traces/analysis/cluster37.oracleGeneral.zst_analysis.json
Pr

In [20]:
df['trace_num'] = df['trace_name'].str.extract(r'cluster(\d+)').astype(int)
df = df.sort_values(by='trace_num').reset_index(drop=True)
df = df.drop(columns=['trace_num'])

In [21]:
df['number_of_requests'] = df['number_of_requests']/1000_000

In [22]:
df['slabs'] = df['number_of_obj_GiB'] * (1024 / 4)

In [30]:
def get_aligned_size(size, alignment):
    return (size + alignment - 1) // alignment * alignment


def generate_alloc_sizes(factor, max_size, min_size, alignment=8):
    if max_size > 4 * 1024 * 1024:
        raise ValueError(f"maximum alloc size {max_size} is more than the slab size {1024 * 1024}")

    if factor <= 1.0:
        raise ValueError(f"invalid factor {factor}")

    alloc_sizes = set()
    size = min_size

    while size < max_size:
        n_per_slab = 4 * 1024 * 1024 // size  # Assuming Slab::kSize is 1MB
        if n_per_slab <= 1:
            break
        alloc_sizes.add(size)
        prev_size = size
        size = get_aligned_size(int(size * factor), alignment)
        if prev_size == size:
            raise ValueError(f"invalid incFactor {factor}")

    alloc_sizes.add(get_aligned_size(max_size, alignment))
    return alloc_sizes



In [41]:
def find_nr_classes(row):
    min_size = row['min_obj_size']
    max_size = row['max_obj_size']
    min_size = max(24, min_size) + 60
    max_size = max(24, max_size) + 60
    
    alloc_sizes = generate_alloc_sizes(1.5, max_size, min_size)
    return len(alloc_sizes) < (row['slabs'] * 0.1)


def find_nr_classes(row):
    min_size = row['min_obj_size']
    max_size = row['max_obj_size']
    min_size = max(24, min_size) + 60
    max_size = max(24, max_size) + 60
    
    alloc_sizes = generate_alloc_sizes(1.5, max_size, min_size)
    return len(alloc_sizes)

def calc_min_alloc_size(row):
    min_size = row['min_obj_size']
    min_size = max(24, min_size) + 60
    return min_size

def calc_max_alloc_size(row):
    max_size = row['max_obj_size']
    max_size = max(24, max_size) + 60
    return max_size

df['can_run'] = df.apply(find_nr_classes, axis=1)
df['min_alloc_size'] = df.apply(calc_min_alloc_size, axis=1)
df['max_alloc_size'] = df.apply(calc_max_alloc_size, axis=1)
df['nr_classes'] = df.apply(find_nr_classes, axis=1)
    

In [36]:
24 + 60

84

In [40]:
df

,trace_name,number_of_requests,number_of_objects,number_of_req_GiB,number_of_obj_GiB,compulsory_miss_ratio_req,compulsory_miss_ratio_byte,frequency_mean,time_span,zipf_slope,zipf_intercept,zipf_r2,min_obj_size,max_obj_size,slabs,can_run,min_alloc_size,max_alloc_size,nr_classes
0,cluster2,7226.679214,3580573,819.4018,0.1362,0.0005,0.0002,2018.3024,620014,1.6978,-1,-1,14,24999,34.8672,10,84,25059,10
1,cluster3,820.307312,6265139,80.5174,0.4316,0.0076,0.0054,130.9320,630361,1.6815,-1,-1,8,5442,110.4896,8,84,5502,8
2,cluster4,3448.082328,106578360,770.5402,13.6594,0.0309,0.0177,32.3526,622662,1.1088,-1,-1,56,179665,3496.8064,12,116,179725,12
3,cluster7,1044.513075,4467952,1108.1038,0.5740,0.0043,0.0005,233.7789,625171,1.2495,-1,-1,6,10405,146.9440,8,84,10465,8
4,cluster8,1302.137879,267311,20374.3291,1.0155,0.0002,0.0000,4871.2469,628689,2.0814,-1,-1,17,767375,259.9680,15,84,767435,15
5,cluster11,2731.402190,26619854,1021.0848,3.9633,0.0097,0.0039,102.6077,622634,1.5993,-1,-1,13,67087,1014.6048,11,84,67147,11
6,cluster13,825.381985,518251246,1665.8210,1403.5642,0.6279,0.8426,1.5926,625199,0.4527,-1,-1,44,769391,359312.4352,14,104,769451,14
7,cluster14,3029.143401,69408243,724.3800,6.5559,0.0229,0.0091,43.6424,506548,1.3503,-1,-1,28,134692,1678.3104,12,88,134752,12
8,cluster19,2006.896318,328491889,89.3905,13.3198,0.1637,0.1490,6.1094,627926,0.8367,-1,-1,20,202,3409.8688,3,84,262,3
9,cluster20,3663.259123,89725634,230.9230,6.0970,0.0245,0.0264,40.8273,623898,1.2797,-1,-1,10,55909,1560.8320,11,84,55969,11
